In [ ]:
# Notebook 12e: RF — Real Only (path_mnist)
import os, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
SEED=42; np.random.seed(SEED)
CLASS_NAMES=['Adipose', 'Background', 'Debris', 'Lymphocytes', 'Mucus', 'Smooth Muscle', 'Normal Colon Mucosa', 'Cancer-Assoc. Stroma', 'Colorectal Adenocarcinoma']
CRITERION='entropy'; MAX_DEPTH=50; N_ESTIMATORS=100
PROJECT_DIR='/content/drive/MyDrive/thesis_project'
REAL_PATH=os.path.join(PROJECT_DIR,'data','processed','seed_42','path_mnist.npz')
RESULTS_DIR=os.path.join(PROJECT_DIR,'results','rf','path_mnist','real_only')
os.makedirs(RESULTS_DIR,exist_ok=True)

In [ ]:
real=np.load(REAL_PATH)
X_tr=np.concatenate([real['X_train'],real['X_val']],axis=0)
y_tr=np.concatenate([real['y_train'],real['y_val']],axis=0)
X_test=real['X_test']; y_test=real['y_test']
print(f'Train: {X_tr.shape}, Test: {X_test.shape}')

In [ ]:
def preprocess(X): return X.astype('float32').reshape(len(X),-1)/255.0
X_tr_n=preprocess(X_tr); X_test_n=preprocess(X_test)

In [ ]:
clf=RandomForestClassifier(criterion=CRITERION,max_depth=MAX_DEPTH,
    n_estimators=N_ESTIMATORS,random_state=SEED,n_jobs=-1)
clf.fit(X_tr_n,y_tr)
y_pred=clf.predict(X_test_n)
test_acc=accuracy_score(y_test,y_pred)
print(f'Test accuracy: {test_acc:.4f}')
print(classification_report(y_test,y_pred,target_names=CLASS_NAMES,digits=4))
report_dict = classification_report(y_test, y_pred,
    target_names=CLASS_NAMES, digits=4, output_dict=True)
macro_f1    = round(report_dict['macro avg']['f1-score'], 4)
weighted_f1 = round(report_dict['weighted avg']['f1-score'], 4)
pd.DataFrame(report_dict).transpose().to_csv(
    os.path.join(RESULTS_DIR, 'classification_report.csv'))
pd.DataFrame([{
    'condition':   'real_only',
    'dataset':     'path_mnist',
    'test_acc':    round(test_acc, 4),
    'macro_f1':    macro_f1,
    'weighted_f1': weighted_f1,
}]).to_csv(os.path.join(RESULTS_DIR, 'summary.csv'), index=False)

In [ ]:
# ── Final Cell: Save Predictions & Confusion Matrix ──────────────────────────
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

y_prob = clf.predict_proba(X_test_n)
y_pred = clf.predict(X_test_n)

np.save(os.path.join(RESULTS_DIR, 'y_pred.npy'), y_pred)
np.save(os.path.join(RESULTS_DIR, 'y_test.npy'), y_test)
np.save(os.path.join(RESULTS_DIR, 'y_prob.npy'), y_prob)

cm = confusion_matrix(y_test, y_pred)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    os.path.join(RESULTS_DIR, 'confusion_matrix.csv'))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — path_mnist | real_only')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.close()
print('All outputs saved.')